# Figure S16 companion: tests for the Jiang comparison groups

Tables only (the corresponding figures are S12b and S16a):

1. Kruskal-Wallis tests of GC content and coverage across the four groups, plus pairwise Mann-Whitney tests ->
   `figS16_jiang_group_scalar_tests.csv`.
2. AUROC of HCC against each comparison group for the Cristiano model, GC alone (lower = HCC), coverage alone
   (higher = HCC) and the model trained within Jiang (pooled out-of-fold, `hcc_vs_nonhcc`), each with a 95%
   stratified participant-level bootstrap interval (2,000 resamples) -> `figS16_jiang_group_aurocs.csv`.
   S16a shows GC in the direction learned on the Cristiano cohort, i.e. 1 - AUROC.
3. Among the 135 non-HCC participants, OLS of the posterior log-odds on liver disease (vs healthy), unadjusted
   and adjusted for GC and coverage, with HC3 standard errors, for both models ->
   `figS16_jiang_group_regression.csv`.

Sources: `data/no_deep/figS16_jiang_scores.csv` (Cristiano model, column `tss`), `data/figS10_jiang_scalars.csv`
(GC, coverage), `data/no_deep/fig4gi_jiang_nested_oof.csv` (model trained within Jiang). Outputs go to `output/no_deep/`.

In [1]:
import os

if not os.path.isdir("data") and os.path.isdir(os.path.join("src", "figures", "data")):
    os.chdir(os.path.join("src", "figures"))
COHORT = os.environ.get("FIG_COHORT", "no_deep")
assert COHORT == "no_deep", "the Jiang per-group analysis is defined on the no_deep Cristiano model"
_DATA, _OUT = os.path.join("data", COHORT), os.path.join("output", COHORT)
os.makedirs(_OUT, exist_ok=True)
out_path = lambda name: os.path.join(_OUT, name)

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu, norm
from sklearn.metrics import roc_auc_score

GROUPS = ["Healthy", "Chronic hepatitis B", "HBV-associated cirrhosis", "HCC"]
SHORT = {"Healthy": "Healthy", "Chronic hepatitis B": "Chronic HBV", "HBV-associated cirrhosis": "Cirrhosis", "HCC": "HCC"}
EPS = 1e-4
logit = lambda p: np.log(np.clip(p, EPS, 1 - EPS) / (1 - np.clip(p, EPS, 1 - EPS)))

j = pd.read_csv(os.path.join(_DATA, "figS16_jiang_scores.csv"))[["sample_id", "y_true", "control_group", "tss"]]
sc = pd.read_csv(os.path.join(_DATA, "figS10_jiang_scalars.csv"))[["sample_id", "mean_gc", "mean_coverage", "sex_bam"]]
oof = pd.read_csv(os.path.join(_DATA, "fig4gi_jiang_nested_oof.csv"))
oof = oof[oof["task"] == "hcc_vs_nonhcc"][["sample_id", "score"]].rename(columns={"score": "within"})
d = j.merge(sc, on="sample_id").merge(oof, on="sample_id")
d["group"] = np.where(d["y_true"] == 1, "HCC", d["control_group"])
d["lo_cris"], d["lo_within"] = logit(d["tss"]), logit(d["within"])
assert len(d) == 225 and set(d["group"]) == set(GROUPS), d["group"].value_counts()
print(d.groupby("group")[["mean_gc", "mean_coverage", "tss", "within"]].median().loc[GROUPS].round(3))

                          mean_gc  mean_coverage    tss  within
group                                                          
Healthy                    41.482          2.243  0.190   0.030
Chronic hepatitis B        42.001          1.876  0.105   0.008
HBV-associated cirrhosis   42.070          1.830  0.062   0.010
HCC                        41.165          2.152  0.326   0.990


In [2]:
# a) are the groups technically different? Kruskal-Wallis across the four groups, pairwise Mann-Whitney
rows = []
for v in ["mean_gc", "mean_coverage"]:
    rows.append(dict(measure=v, comparison="all four groups (Kruskal-Wallis)",
                     p=kruskal(*[d.loc[d.group == g, v] for g in GROUPS]).pvalue))
    for a_, b_ in [("Healthy", "HCC"), ("Healthy", "Chronic hepatitis B"), ("Healthy", "HBV-associated cirrhosis"),
                   ("HCC", "Chronic hepatitis B"), ("HCC", "HBV-associated cirrhosis")]:
        rows.append(dict(measure=v, comparison=f"{SHORT[a_]} vs {SHORT[b_]} (Mann-Whitney)",
                         p=mannwhitneyu(d.loc[d.group == a_, v], d.loc[d.group == b_, v]).pvalue))
tests = pd.DataFrame(rows)
tests.to_csv(out_path("figS16_jiang_group_scalar_tests.csv"), index=False)
print(tests.to_string(index=False, formatters={"p": "{:.2g}".format}))

      measure                            comparison       p
      mean_gc      all four groups (Kruskal-Wallis) 1.8e-16
      mean_gc         Healthy vs HCC (Mann-Whitney)    0.09
      mean_gc Healthy vs Chronic HBV (Mann-Whitney) 7.6e-08
      mean_gc   Healthy vs Cirrhosis (Mann-Whitney) 3.3e-08
      mean_gc     HCC vs Chronic HBV (Mann-Whitney) 1.1e-11
      mean_gc       HCC vs Cirrhosis (Mann-Whitney) 1.1e-09
mean_coverage      all four groups (Kruskal-Wallis)  0.0054
mean_coverage         Healthy vs HCC (Mann-Whitney)    0.57
mean_coverage Healthy vs Chronic HBV (Mann-Whitney)   0.014
mean_coverage   Healthy vs Cirrhosis (Mann-Whitney)   0.017
mean_coverage     HCC vs Chronic HBV (Mann-Whitney)   0.009
mean_coverage       HCC vs Cirrhosis (Mann-Whitney)   0.016


In [3]:
# b) AUROC of HCC against each comparison group, four scores, stratified participant-level bootstrap
SCORES = {"Cristiano model": d["tss"], "GC content alone (lower = HCC)": -d["mean_gc"],
          "coverage alone (higher = HCC)": d["mean_coverage"], "model trained within Jiang": d["within"]}
rng = np.random.default_rng(0)
rows = []
for cg in ["Healthy", "Chronic hepatitis B", "HBV-associated cirrhosis"]:
    pos, neg = np.where(d.group == "HCC")[0], np.where(d.group == cg)[0]
    lab = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    draws = [(rng.choice(pos, len(pos)), rng.choice(neg, len(neg))) for _ in range(2000)]
    for name, s in SCORES.items():
        s = s.to_numpy()
        bs = [roc_auc_score(lab, np.r_[s[p], s[n]]) for p, n in draws]
        rows.append(dict(comparison=f"HCC vs {cg}", n_hcc=len(pos), n_control=len(neg), score=name,
                         auroc=roc_auc_score(lab, np.r_[s[pos], s[neg]]), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
aur = pd.DataFrame(rows)
aur.round(4).to_csv(out_path("figS16_jiang_group_aurocs.csv"), index=False)
print(aur.pivot(index="comparison", columns="score", values="auroc").round(3).to_string())

score                            Cristiano model  GC content alone (lower = HCC)  coverage alone (higher = HCC)  model trained within Jiang
comparison                                                                                                                                 
HCC vs Chronic hepatitis B                 0.776                           0.817                          0.622                       0.869
HCC vs HBV-associated cirrhosis            0.791                           0.849                          0.638                       0.897
HCC vs Healthy                             0.612                           0.601                          0.466                       0.864


In [4]:
# c, d) regression among the 135 non-HCC participants: log-odds ~ liver disease (+ GC + coverage), HC3 errors
def ols_hc3(X, y):
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    hat = np.einsum("ij,jk,ik->i", X, XtX_inv, X)
    se = np.sqrt(np.diag(XtX_inv @ (X.T * (e / (1 - hat)) ** 2) @ X @ XtX_inv))
    return b, se

c = d[d.y_true == 0].copy()
c["liver"] = (c.group != "Healthy").astype(float)
gc_m, cov_m = c.mean_gc.mean(), c.mean_coverage.mean()
rows, fits = [], {}
for model, col in [("Cristiano model", "lo_cris"), ("model trained within Jiang", "lo_within")]:
    for adj, cols, names in [("unadjusted", [c.liver], ["liver disease vs healthy"]),
                             ("adjusted for GC and coverage", [c.liver, c.mean_gc - gc_m, c.mean_coverage - cov_m],
                              ["liver disease vs healthy", "GC content (per %)", "coverage (per 1x)"])]:
        X = np.column_stack([np.ones(len(c))] + [np.asarray(x, float) for x in cols])
        b, se = ols_hc3(X, c[col].to_numpy())
        fits[(model, adj)] = b
        for nm, bi, si in zip(["intercept"] + names, b, se):
            rows.append(dict(model=model, adjustment=adj, term=nm, estimate=bi, se_hc3=si, ci_lo=bi - 1.96 * si,
                             ci_hi=bi + 1.96 * si, p=2 * norm.sf(abs(bi / si))))
reg = pd.DataFrame(rows)
reg.round(4).to_csv(out_path("figS16_jiang_group_regression.csv"), index=False)
print(reg[reg.term != "intercept"].round(3).to_string(index=False))

                     model                   adjustment                     term  estimate  se_hc3  ci_lo  ci_hi     p
           Cristiano model                   unadjusted liver disease vs healthy    -1.145   0.232 -1.599 -0.691 0.000
           Cristiano model adjusted for GC and coverage liver disease vs healthy    -0.716   0.240 -1.187 -0.244 0.003
           Cristiano model adjusted for GC and coverage       GC content (per %)    -0.474   0.247 -0.959  0.011 0.056
           Cristiano model adjusted for GC and coverage        coverage (per 1x)     0.575   0.256  0.073  1.076 0.025
model trained within Jiang                   unadjusted liver disease vs healthy    -0.524   0.917 -2.322  1.274 0.568
model trained within Jiang adjusted for GC and coverage liver disease vs healthy     0.153   1.185 -2.169  2.476 0.897
model trained within Jiang adjusted for GC and coverage       GC content (per %)    -0.518   0.808 -2.102  1.067 0.522
model trained within Jiang adjusted for GC and c

## Within each sex

The Cristiano model's AUROC (HCC versus all non-HCC) within the men and within the women of the
Jiang cohort, sex called from the reads (`sex_bam`, as in Fig S16).

In [5]:
for sex, g in d.groupby("sex_bam"):
    print(f"{sex}: n = {len(g)} ({int(g['y_true'].sum())} HCC), AUROC {roc_auc_score(g['y_true'], g['tss']):.3f}")
print(f"all: n = {len(d)}, AUROC {roc_auc_score(d['y_true'], d['tss']):.3f}")

female: n = 63 (15 HCC), AUROC 0.738
male: n = 162 (75 HCC), AUROC 0.744
all: n = 225, AUROC 0.741
